# K-Means: From Intuition to Algorithm 🔍
### Machine Learning School · Hands-on notebook

**Learning goals**
1. Discover what clustering means when we have **no labels**.
2. Understand the two iterative steps of K-Means: **assignment** and **centroid update**.
3. Connect the algorithm to its objective: **Within-Cluster Sum of Squares (WCSS)**.
4. Explore how the choice of **K**, feature scaling, and cluster geometry affect results.

**Estimated duration:** 35–50 minutes · **Requirements:** `numpy`, `matplotlib`, `scikit-learn` (and `pandas` for one small table).

> **Key idea:** K-Means does not know the “true” groups. It partitions the data by minimizing squared distances to cluster centroids.


## 0 · Setup
Run this cell first. All examples are reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import ListedColormap

SEED = 42
rng = np.random.default_rng(SEED)
COLORS = ['#ED247B', '#2785B5', '#199A83', '#E8A343', '#8B65B9', '#70757E']
plt.rcParams.update({'figure.figsize': (8, 5), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print("Ready! ✨")

## 1 · Can you see the clusters?

We start with unlabeled observations. **Look at the scatter plot before running K-Means.**

**Think:** How many groups do you see? What makes two points “similar”?

*Note:* We generate synthetic groups to build the exercise, but the clustering algorithm receives **only the coordinates**, not the labels used to generate them.


In [ ]:
#make_blobs generates artificial groups of data points, where each group has a roughly circular shape and 
# the points are distributed around a central location.
# Check: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html 
X, hidden_labels = make_blobs(
    n_samples=240, 
    centers=[(-4, -1), (0, 4), (4, -1)],
    cluster_std=[0.85, 1.0, 0.95], 
    random_state=SEED
)
fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], c='#8A8D95', s=28, alpha=.8)
ax.set(xlabel='Feature 1', ylabel='Feature 2', title='Unlabeled data: what patterns do you see?')
ax.set_aspect('equal', adjustable='datalim')
plt.show()

## 2 · K-Means in action

For a chosen number of clusters \(K\), K-Means alternates:

1. **Assignment:** assign each observation to the nearest centroid.
2. **Update:** move each centroid to the mean of its assigned observations.

Repeat until assignments stop changing (or another stopping criterion is reached).

### Explore: choose K
Change `K` to 2, 3, 4, or 5 and rerun. Compare the cluster shapes and centroids.


In [ ]:
K = 3  #  Try 2, 3, 4, or 5

model = KMeans(n_clusters=K, init='k-means++', n_init=10, random_state=SEED)
labels = model.fit_predict(X)
centers = model.cluster_centers_

fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], c=labels,
           cmap=ListedColormap(COLORS[:K]), s=28, alpha=.75)
ax.scatter(centers[:, 0], centers[:, 1], c='white',
           edgecolors='black', linewidths=2, marker='X', s=230, label='Centroids')
ax.set(xlabel='Feature 1', ylabel='Feature 2', title=f'K-Means with K = {K}')
ax.legend()
plt.show()
print(f"WCSS (inertia): {model.inertia_:,.1f}")

In [ ]:
X_demo.shape

## 3 · See the iterations (without calling `KMeans`)

Let's implement the two steps ourselves to see how the centroids move.

We use a small subset of points and **fixed initial centroids** for visualization. 
This is a teaching implementation; scikit-learn handles additional details such as empty clusters and stopping criteria.


In [ ]:
# Select every third row and make a copy
X_demo = X[::3].copy()
initial_centers = np.array([[-5.5, -2.0], [-1.0, 2.2], [5.4, 0.8]])

# Assign each point to its nearest centroid
def assign_points(data, centroids):
    # Compute squared Euclidean distances between all points and centroids
    # None adds a dimension to enable NumPy broadcasting
    squared_distances = ((data[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    # Return the index of the nearest centroid for each point
    return squared_distances.argmin(axis=1)

# Update each centroid as the mean of its assigned points
def update_centroids(data, labels, old_centroids):
    new = old_centroids.copy()
     # Iterate over all clusters
    for k in range(len(old_centroids)):
        # Check whether the cluster contains any points
        if np.any(labels == k):
            # Select points assigned to cluster k and compute their mean
            new[k] = data[labels == k].mean(axis=0)
    return new

# Compute Within-Cluster Sum of Squares (WCSS)
def wcss(data, labels, centroids):
    # Sum squared distances from each point to its assigned centroid
    return np.sum((data - centroids[labels]) ** 2)

# Initialize centroids and store the history of iterations
centroids = initial_centers.copy()
history = []
# Run K-Means for 7 iterations
for iteration in range(7):
    assignment = assign_points(X_demo, centroids) # Assignment step
    objective = wcss(X_demo, assignment, centroids) # Compute WCSS
     # Save centroids, assignments, and WCSS for visualization
    history.append((centroids.copy(), assignment.copy(), objective))
    centroids = update_centroids(X_demo, assignment, centroids) # Update step

# Visualize the first 6 iterations in a 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(13, 8), sharex=True, sharey=True)
for ax, (iteration, snapshot) in zip(axes.flat, enumerate(history[:6])):
    # Retrieve the saved state of the current iteration
    c, lab, score = snapshot
     # Plot points colored by their assigned cluster
    ax.scatter(X_demo[:, 0], X_demo[:, 1], c=lab,
               cmap=ListedColormap(COLORS[:3]), s=20, alpha=.7)
    # Plot centroids as X markers
    ax.scatter(c[:, 0], c[:, 1], marker='X', c='white',
               edgecolors='black', s=160, linewidths=1.6)
    # Display iteration number and WCSS
    ax.set_title(f'Iteration {iteration} · WCSS = {score:.0f}')
for ax in axes[-1]:
    ax.set_xlabel('Feature 1')
for ax in axes[:, 0]:
    ax.set_ylabel('Feature 2')
fig.suptitle('Assignment → update → assignment → ...', fontsize=15)
plt.tight_layout()
plt.show()

**Notice:** The centroids move toward the observations assigned to them. WCSS generally decreases as the algorithm progresses.

### Why WCSS?

For cluster \(C_k\), let its centroid be

$$\boldsymbol{\mu}_k=\frac{1}{|C_k|}\sum_{i\in C_k}\mathbf{x}_i. $$

The **Within-Cluster Sum of Squares** is


$$\boxed{\mathrm{WCSS}=\sum_{k=1}^{K}\sum_{i\in C_k}\|\mathbf{x}_i-\boldsymbol{\mu}_k\|_2^2.}$$


Smaller WCSS means points are, on average, closer to their assigned centroids. **But:** increasing \(K\) will almost always reduce WCSS, so the smallest WCSS alone is not a good way to choose \(K\).


In [ ]:
scores = [h[2] for h in history]
plt.figure(figsize=(7, 3.5))
plt.plot(range(len(scores)), scores, 'o-', color=COLORS[0], linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('WCSS')
plt.title('Objective during the manual iterations')
plt.grid(alpha=.2)
plt.show()


## 4 · How many clusters? Elbow + Silhouette

How do we choose the number of clusters, **K**? We explore two common approaches.

### Elbow Method

The **Within-Cluster Sum of Squares (WCSS)** measures how close observations are to their assigned centroids:

$$
\mathrm{WCSS} = \sum_{k=1}^{K}\sum_{i\in C_k}
\|\mathbf{x}_i-\boldsymbol{\mu}_k\|_2^2
$$

- Lower WCSS indicates more compact clusters.
- WCSS decreases (or stays the same) as K increases.
- We look for an **elbow**: the point where adding more clusters produces only a small improvement.

### Silhouette Score

The **Silhouette Score** measures how well each observation fits within its assigned cluster compared to other clusters.

For each observation $i$, define:

- $a(i)$: average distance to other observations in the **same cluster**.
- $b(i)$: smallest average distance to observations in **another cluster**.

The silhouette coefficient is:

$$
s(i) = \frac{b(i)-a(i)}{\max(a(i),b(i))}
$$

The overall Silhouette Score is the average across observations.

**Interpretation:**

- **Close to +1:** well-separated clusters.
- **Close to 0:** overlapping clusters.
- **Close to −1:** observations may be assigned to the wrong cluster.

Higher scores generally indicate better-defined clusters.

> **Remember:** The Elbow Method evaluates cluster compactness, while the Silhouette Score evaluates both cohesion and separation. Neither method guarantees the "true" number of clusters.



In [ ]:
# Test different numbers of clusters (K = 2, ..., 8)
ks = list(range(2, 9))
inertias, silhouettes = [], []

# Store WCSS (inertia) and silhouette scores for each K
for k in ks:
    # Initialize K-Means with K clusters and 10 different initializations
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)

    # Fit K-Means and assign each observation to a cluster
    predicted = km.fit_predict(X)

    # Store WCSS: sum of squared distances to assigned centroids
    inertias.append(km.inertia_)

    # Store silhouette score: measures cluster cohesion and separation
    silhouettes.append(silhouette_score(X, predicted))

# Create two plots to compare the methods
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, inertias, 'o-', color=COLORS[0])

# Silhouette analysis: higher scores indicate better-defined clusters
axes[0].set(xlabel='Number of clusters K', ylabel='WCSS', title='Elbow method')
axes[1].plot(ks, silhouettes, 'o-', color=COLORS[1])
axes[1].set(xlabel='Number of clusters K', ylabel='Silhouette score', title='Silhouette analysis')

# Format both plots
for ax in axes:
    ax.set_xticks(ks)
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

# Select K with the highest silhouette score
# np.argmax returns the index of the maximum score
print('Best K by silhouette in this example:', ks[int(np.argmax(silhouettes))])

## 5 · A subtle problem: feature scales matter

K-Means uses Euclidean distances. A feature measured on a much larger numerical scale can dominate those distances.

We deliberately multiply one feature by 100. Compare the clusters **before and after standardization**.


In [ ]:
X_stretched = X.copy()
X_stretched[:, 0] *= 100
X_scaled = StandardScaler().fit_transform(X_stretched)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in zip(
    axes, [X_stretched, X_scaled],
    ['Raw features (feature 1 × 100)', 'After StandardScaler']
):
    pred = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(data)
    ax.scatter(data[:, 0], data[:, 1], c=pred,
               cmap=ListedColormap(COLORS[:3]), s=22, alpha=.8)
    ax.set(title=title, xlabel='Feature 1', ylabel='Feature 2')
plt.tight_layout()
plt.show()
print('Question: why can the assignments change even though we did not change the observations?')

## 6 · When K-Means struggles: non-spherical clusters

K-Means is especially natural for compact, roughly spherical groups. It can struggle with curved or irregular shapes.

Try the **two moons** dataset. Compare K-Means with DBSCAN (a density-based method). The synthetic labels are used only to evaluate the result, **not to fit either model**.


In [ ]:
X_moons, y_moons = make_moons(n_samples=350, noise=.075, random_state=SEED)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(X_moons)
db_moons = DBSCAN(eps=.20, min_samples=5).fit_predict(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True, sharey=True)
for ax, lab, title in zip(
    axes,
    [y_moons, km_moons, db_moons],
    ['Synthetic reference groups', 'K-Means (K = 2)', 'DBSCAN']
):
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=lab,
               cmap=ListedColormap(COLORS[:3]), s=16, alpha=.85)
    ax.set_title(title)
    ax.set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()
print('ARI vs. synthetic reference labels (1 = exact agreement):')
print('K-Means:', round(adjusted_rand_score(y_moons, km_moons), 3))
print('DBSCAN: ', round(adjusted_rand_score(y_moons, db_moons), 3))
print('DBSCAN uses -1 for points considered noise, if any.')